In [1]:
!pip install -q kagglehub fastapi uvicorn pydantic joblib pytest httpx requests gradio evidently locust

import os, pathlib
import pandas as pd
import numpy as np
import kagglehub
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# Создаем папки прямо в Colab
PROJECT = '/content/lab2_project'
for sub in ['src', 'tests', 'data', 'models', 'reports']:
    pathlib.Path(f'{PROJECT}/{sub}').mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)

# Скачиваем данные
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
file_path = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(file_path)

# Чистим
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop(columns=['customerID'])

X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_test.to_csv('data/X_test.csv', index=False)

# Пайплайн
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])
model_pipeline = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

model_pipeline.fit(X_train, y_train)
proba = model_pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)

joblib.dump(model_pipeline, 'models/model.pkl')
print(f"✅ Модель обучена! ROC-AUC: {auc:.4f}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Using Colab cache for faster access to the 'telco-customer-churn' dataset.
✅ Модель обучена! ROC-AUC: 0.8375


In [2]:
%%writefile src/service.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI(title="Churn Prediction Service")
model = joblib.load('models/model.pkl')
THRESHOLD = 0.12

class CustomerData(BaseModel):
    gender: str = "Female"
    SeniorCitizen: int = 0
    Partner: str = "No"
    Dependents: str = "No"
    tenure: int = 0
    PhoneService: str = "Yes"
    MultipleLines: str = "No"
    InternetService: str = "DSL"
    OnlineSecurity: str = "No"
    OnlineBackup: str = "No"
    DeviceProtection: str = "No"
    TechSupport: str = "No"
    StreamingTV: str = "No"
    StreamingMovies: str = "No"
    Contract: str = "Month-to-month"
    PaperlessBilling: str = "Yes"
    PaymentMethod: str = "Electronic check"
    MonthlyCharges: float = 0.0
    TotalCharges: float = 0.0

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(data: CustomerData):
    try:
        df = pd.DataFrame([data.model_dump()])
        proba = float(model.predict_proba(df)[0, 1])
        return {"proba": proba, "decision": int(proba >= THRESHOLD)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Writing src/service.py


In [3]:
import os, uvicorn, threading, time, requests

!kill -9 $(lsof -t -i:8000) 2>/dev/null || true
time.sleep(2)
os.chdir('/content/lab2_project')

def run_server():
    uvicorn.run("src.service:app", host="127.0.0.1", port=8000, log_level="error")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(5)

r = requests.post('http://127.0.0.1:8000/predict', json={"tenure": 2, "MonthlyCharges": 80.0})
print("Статус:", r.status_code)
print("Ответ:", r.json())

Статус: 200
Ответ: {'proba': 0.5493641684329716, 'decision': 1}


In [4]:
%%writefile tests/test_service.py
from fastapi.testclient import TestClient
from src.service import app
client = TestClient(app)

OK_DATA = {"tenure": 12, "MonthlyCharges": 70.5}

def test_health():
    assert client.get('/health').json()['status'] == 'ok'

def test_predict_ok():
    r = client.post('/predict', json=OK_DATA)
    assert r.status_code == 200

def test_wrong_type():
    assert client.post('/predict', json={"tenure": "строка", "MonthlyCharges": 70.5}).status_code == 422

def test_out_of_range():
    assert client.post('/predict', json={"tenure": -5, "MonthlyCharges": 70.5}).status_code == 422

Writing tests/test_service.py


In [6]:
%%writefile src/service.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field  # <--- Добавили Field
import pandas as pd
import joblib

app = FastAPI(title="Churn Prediction Service")
model = joblib.load('models/model.pkl')
THRESHOLD = 0.12

class CustomerData(BaseModel):
    gender: str = "Female"
    SeniorCitizen: int = 0
    Partner: str = "No"
    Dependents: str = "No"
    # ЗАПРЕЩАЕМ отрицательный стаж (ge=0)
    tenure: int = Field(0, ge=0)  # <--- ВОТ ИСПРАВЛЕНИЕ
    PhoneService: str = "Yes"
    MultipleLines: str = "No"
    InternetService: str = "DSL"
    OnlineSecurity: str = "No"
    OnlineBackup: str = "No"
    DeviceProtection: str = "No"
    TechSupport: str = "No"
    StreamingTV: str = "No"
    StreamingMovies: str = "No"
    Contract: str = "Month-to-month"
    PaperlessBilling: str = "Yes"
    PaymentMethod: str = "Electronic check"
    MonthlyCharges: float = 0.0
    TotalCharges: float = 0.0

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(data: CustomerData):
    try:
        df = pd.DataFrame([data.model_dump()])
        proba = float(model.predict_proba(df)[0, 1])
        return {"proba": proba, "decision": int(proba >= THRESHOLD)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Overwriting src/service.py


In [7]:
!python -m pytest tests/ -q

....                                                                     [100%]
4 passed in 3.03s


In [8]:
%%writefile locustfile.py
from locust import HttpUser, task, between
OK_DATA = {"tenure": 12, "MonthlyCharges": 70.5}
class QuickstartUser(HttpUser):
    wait_time = between(0.01, 0.05)
    @task
    def predict(self):
        self.client.post("/predict", json=OK_DATA)

Writing locustfile.py


In [9]:
!locust -f locustfile.py --headless -u 20 -r 5 -t 20s --host http://127.0.0.1:8000 --csv reports/load

[2026-11-04 15:04:50,475] c77c0f0cb81c/INFO/locust.main: Starting Locust 2.47.0
[2026-11-04 15:04:50,476] c77c0f0cb81c/INFO/locust.main: Run time limit set to 20 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-11-04 15:04:50,478] c77c0f0cb81c/INFO/locust.runners: Ramping to 20 users at a rate of 5.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict      48     0(0.00%) |    232      62     453    210 |    0.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggre

In [11]:
!pip install -q "evidently==0.4.28"

In [2]:
import os, sys, uvicorn, threading, time, requests
import gradio as gr

# 1. Переходим в папку проекта
os.chdir('/content/lab2_project')
sys.path.insert(0, '/content/lab2_project')

# 2. Убиваем старый сервер, если он висит
!kill -9 $(lsof -t -i:8000) 2>/dev/null || true
time.sleep(2)

# 3. Запускаем сервер в фоне
def run_server():
    uvicorn.run("src.service:app", host="127.0.0.1", port=8000, log_level="error")

threading.Thread(target=run_server, daemon=True).start()
print("Ждем запуска сервера (5 сек)...")
time.sleep(5)

# 4. Функция для Gradio
def predict_churn(tenure, monthly):
    try:
        r = requests.post('http://127.0.0.1:8000/predict', json={"tenure": int(tenure), "MonthlyCharges": float(monthly)}, timeout=5)
        if r.status_code != 200:
            return f"Ошибка сервера: {r.text}"
        res = r.json()
        return f"Вероятность: {res['proba']:.2f} | Решение: {'Уйдёт' if res['decision'] else 'Останется'}"
    except Exception as e:
        return f"Ошибка подключения: {e}"

# 5. Запускаем Gradio
demo = gr.Interface(
    fn=predict_churn,
    inputs=[
        gr.Number(label="Стаж (месяцев)", value=12),
        gr.Number(label="Ежемесячный платеж", value=70.5)
    ],
    outputs=gr.Textbox(label="Результат"),
    title="Оценка риска оттока (ЛР2)"
)
demo.launch(share=True)

Ждем запуска сервера (5 сек)...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e893a71511f24124ae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import requests
import json

# Делаем запрос к нашему серверу
print("Отправляем запрос на /predict...")
r = requests.post('http://127.0.0.1:8000/predict', json={"tenure": 33, "MonthlyCharges": 63})

print("\n--- ОТВЕТ СЕРВИСА ---")
print(f"HTTP Статус: {r.status_code}")
print(f"JSON ответ: {json.dumps(r.json(), indent=4, ensure_ascii=False)}")

Отправляем запрос на /predict...

--- ОТВЕТ СЕРВИСА ---
HTTP Статус: 200
JSON ответ: {
    "proba": 0.387535779371997,
    "decision": 1
}


In [4]:
!python -m pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/lab2_project
plugins: Faker-40.44.0, locust-2.47.0, langsmith-0.14.15, typeguard-4.6.0, platformdirs-4.14.0, anyio-4.15.1
collected 4 items                                                              

tests/test_service.py::test_health PASSED                                [ 25%]
tests/test_service.py::test_predict_ok PASSED                            [ 50%]
tests/test_service.py::test_wrong_type PASSED                            [ 75%]
tests/test_service.py::test_out_of_range PASSED                          [100%]

============================== 4 passed in 4.46s ===============================


In [5]:
!locust -f locustfile.py --headless -u 20 -r 5 -t 20s --host http://127.0.0.1:8000 --csv reports/load

[2026-11-04 15:18:46,418] c77c0f0cb81c/INFO/locust.main: Starting Locust 2.47.0
[2026-11-04 15:18:46,423] c77c0f0cb81c/INFO/locust.main: Run time limit set to 20 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-11-04 15:18:46,428] c77c0f0cb81c/INFO/locust.runners: Ramping to 20 users at a rate of 5.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict      31     0(0.00%) |    324     150     561    290 |    0.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggre

In [6]:
!locust -f locustfile.py --headless -u 5 -r 1 -t 20s --host http://127.0.0.1:8000

[2026-11-04 15:20:04,478] c77c0f0cb81c/INFO/locust.main: Starting Locust 2.47.0
[2026-11-04 15:20:04,484] c77c0f0cb81c/INFO/locust.main: Run time limit set to 20 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-11-04 15:20:04,488] c77c0f0cb81c/INFO/locust.runners: Ramping to 5 users at a rate of 1.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict      14     0(0.00%) |    161      46     389    140 |    0.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggreg

In [ ]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Загружаем тестовые данные (эталон)
reference = pd.read_csv('data/X_test.csv')

# Создаем "новые" данные с дрифтом (увеличиваем платеж на 20%)
current = reference.copy()
current['MonthlyCharges'] = current['MonthlyCharges'] * 1.2

# Строим отчет
report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=reference, current_data=current)

# Сохраняем в HTML
report.save_html('reports/drift_report.html')
print("✅ Отчет о дрейфе сохранен в reports/drift_report.html")

In [7]:
!locust -f locustfile.py --headless -u 50 -r 5 -t 20s --host http://127.0.0.1:8000

[2026-11-04 15:20:37,601] c77c0f0cb81c/INFO/locust.main: Starting Locust 2.47.0
[2026-11-04 15:20:37,604] c77c0f0cb81c/INFO/locust.main: Run time limit set to 20 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-11-04 15:20:37,609] c77c0f0cb81c/INFO/locust.runners: Ramping to 50 users at a rate of 5.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict      25     0(0.00%) |    476     279     760    430 |    0.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggre

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict     233     0(0.00%) |   1011     279    2114   1000 |   19.38        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated     233     0(0.00%) |   1011     279    2114   1000 |   19.38        0.00



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict     299     0(0.00%) |   1144     279    2727   1100 |   21.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated     299     0(0.00%) |   1144     279    2727   1100 |   21.00        0.00



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

... (4 outputs left out)

Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict     466     0(0.00%) |   1335     279    3493   1200 |   28.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated     466     0(0.00%) |   1335     279    3493   1200 |   28.00        0.00

[2026-11-04 15:20:56,090] c77c0f0cb81c/INFO/locust.main: --run-time limit reached, shutting down


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[2026-11-04 15:20:56,155] c77c0f0cb81c/INFO/locust.main: Shutting down (exit code 0)
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict     494     0(0.00%) |   1335     279    3493   1200 |   26.91        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated     494     0(0.00%) |   1335     279    3493   1200 |   26.91        0.00

Response time percentiles (approximated)
Type     Name      50%    66%    75%    80%    90%    95%    98%    99%  99.9% 99.99%   100% # reqs
--------||--------|------|------|------|------|------|------|------|------|------|------|------
POST     /predict     1200   1400   1600   1800   2100   2400   2800   2900   3500   3500   3500    494
--------||--------|------|------|------|------|------|------|------|------|------|------|------
         Aggregated     1200  

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [8]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

# Загружаем тестовые данные (эталон)
reference = pd.read_csv('data/X_test.csv')

# Создаем "новые" данные с дрифтом (увеличиваем платеж на 20%)
current = reference.copy()
current['MonthlyCharges'] = current['MonthlyCharges'] * 1.2

# Строим отчет
report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=reference, current_data=current)

# Сохраняем в HTML
report.save_html('reports/drift_report.html')
print("✅ Отчет о дрейфе сохранен в reports/drift_report.html")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ Отчет о дрейфе сохранен в reports/drift_report.html


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

In [1]:
!pip freeze > requirements.txt
print("✅ Файл requirements.txt создан в корне проекта!")

✅ Файл requirements.txt создан в корне проекта!
